# 7.13 — Anchors, IoU, NMS & mAP

Object detection turns an image into scored boxes, but raw boxes are messy: many overlap, some are duplicates, and evaluation must decide which predictions truly match objects. In this lesson, we build the geometry and ranking machinery from scratch — anchors assign responsibility, IoU measures localization, NMS removes duplicate echoes, and AP/mAP scores the whole confidence ranking.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build the detection toolkit one idea at a time. Run each cell in order and inspect the printed numbers and plots — every formula is written as simple NumPy so the matching, suppression, and AP logic stays visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, geometry, sorting, and cumulative sums.
import matplotlib.pyplot as plt  # visual checks for boxes and precision-recall curves.
np.random.seed(0)  # reproducibility for any random demos.

### 1. Box geometry and IoU

A detection box is easiest to reason about in corner form `[x1, y1, x2, y2]`: left, top, right, bottom. Its width is `x2 - x1`, its height is `y2 - y1`, and the intersection of two boxes is another rectangle whose left edge is the larger left edge and whose right edge is the smaller right edge. **Intersection-over-union** then asks what fraction of the total covered area is shared.

In [ ]:
A_w = np.array([0., 0., 3., 3.])  # first 3-by-3 box.
B_w = np.array([1., 1., 4., 4.])  # second 3-by-3 box shifted down-right.
print("A:", A_w, "area:", (A_w[2] - A_w[0]) * (A_w[3] - A_w[1]))
print("B:", B_w, "area:", (B_w[2] - B_w[0]) * (B_w[3] - B_w[1]))

▶ What you'll see: both boxes have area 9, but they are not aligned.

In [ ]:
inter_x1_w = max(A_w[0], B_w[0])  # shared rectangle's left edge.
inter_y1_w = max(A_w[1], B_w[1])  # shared rectangle's top edge.
inter_x2_w = min(A_w[2], B_w[2])  # shared rectangle's right edge.
inter_y2_w = min(A_w[3], B_w[3])  # shared rectangle's bottom edge.
inter_w_w = max(0.0, inter_x2_w - inter_x1_w)  # clamp so non-overlap gives width 0.
inter_h_w = max(0.0, inter_y2_w - inter_y1_w)  # clamp so non-overlap gives height 0.
inter_area_w = inter_w_w * inter_h_w  # area of A ∩ B.
print("intersection rectangle:", [inter_x1_w, inter_y1_w, inter_x2_w, inter_y2_w])
print("intersection area:", inter_area_w)
assert inter_area_w == 4.0

▶ What you'll see: the shared rectangle is `[1, 1, 3, 3]`, so its area is `2×2=4`.

In [ ]:
area_A_w = (A_w[2] - A_w[0]) * (A_w[3] - A_w[1])  # |A|.
area_B_w = (B_w[2] - B_w[0]) * (B_w[3] - B_w[1])  # |B|.
union_w = area_A_w + area_B_w - inter_area_w  # |A ∪ B| = |A| + |B| - |A ∩ B|.
iou_w = inter_area_w / union_w  # shared area divided by total covered area.
print("union:", union_w)
print("IoU:", round(iou_w, 3))
assert round(iou_w, 3) == 0.286

▶ What you'll see: `4/14 = 0.286`, the canonical overlap from the source lesson.

In [ ]:
plt.figure(figsize=(4, 4))
for box_w, color_w, label_w in [(A_w, "steelblue", "A"), (B_w, "darkorange", "B")]:
    plt.gca().add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, lw=3, ec=color_w, label=label_w))
plt.gca().add_patch(plt.Rectangle((inter_x1_w, inter_y1_w), inter_w_w, inter_h_w, color="seagreen", alpha=0.25, label="intersection"))
plt.xlim(-0.5, 4.5); plt.ylim(4.5, -0.5); plt.gca().set_aspect("equal")
plt.title(f"1: IoU = {iou_w:.3f}"); plt.legend(); plt.show()

▶ What you'll see: two shifted squares with only the central green rectangle shared.

*Why it's done this way:* IoU divides by **union**, not by one box's area, because localization has two kinds of error: missing part of the object and covering extra background. Subtracting the intersection once in the union avoids double-counting the shared region, while clamping intersection width/height at zero makes non-overlapping boxes produce IoU 0 instead of a negative area.

### 2. Anchors choose responsibility by best IoU

An anchor is a preset candidate box. During training, a ground-truth object is assigned to the anchor whose shape and position best overlaps it. That responsibility rule is usually “largest IoU,” but ties are real: `argmax` returns the first maximum, so deterministic tie policy affects which anchor receives the positive label.

In [ ]:
gt_w = np.array([1., 1., 3., 3.])  # ground-truth object box.
anchors_w = np.array([[0., 0., 2., 2.],
                      [0., 0., 3., 3.],
                      [1., 1., 4., 4.]])  # three candidate anchors.
print("ground truth:", gt_w)
print("anchors:\n", anchors_w)

▶ What you'll see: three anchors around the same small target.

In [ ]:
def area_w(box_w):
    return max(0.0, box_w[2] - box_w[0]) * max(0.0, box_w[3] - box_w[1])

def iou_one_w(box1_w, box2_w):
    x1_w = max(box1_w[0], box2_w[0]); y1_w = max(box1_w[1], box2_w[1])
    x2_w = min(box1_w[2], box2_w[2]); y2_w = min(box1_w[3], box2_w[3])
    inter_w = max(0.0, x2_w - x1_w) * max(0.0, y2_w - y1_w)
    union_w = area_w(box1_w) + area_w(box2_w) - inter_w
    return 0.0 if union_w == 0 else inter_w / union_w

iou_anchors_w = np.array([iou_one_w(a_w, gt_w) for a_w in anchors_w])
print("anchor IoUs:", np.round(iou_anchors_w, 3))
assert np.allclose(np.round(iou_anchors_w, 3), [0.143, 0.444, 0.444])

▶ What you'll see: anchors 1 and 2 tie at `4/9 ≈ 0.444`.

In [ ]:
best_anchor_w = int(np.argmax(iou_anchors_w))  # NumPy returns the first maximum in a tie.
print("best anchor index:", best_anchor_w)
print("best anchor box:", anchors_w[best_anchor_w])
assert best_anchor_w == 1

▶ What you'll see: index 1 wins because it is the first anchor with the maximum IoU.

In [ ]:
plt.figure(figsize=(4, 4))
plt.gca().add_patch(plt.Rectangle((gt_w[0], gt_w[1]), gt_w[2]-gt_w[0], gt_w[3]-gt_w[1], fill=True, alpha=0.15, color="black", label="GT"))
for k_w, a_w in enumerate(anchors_w):
    plt.gca().add_patch(plt.Rectangle((a_w[0], a_w[1]), a_w[2]-a_w[0], a_w[3]-a_w[1], fill=False, lw=2, ec=["crimson", "seagreen", "orange"][k_w], label=f"anchor {k_w}: {iou_anchors_w[k_w]:.3f}"))
plt.xlim(-0.5, 4.5); plt.ylim(4.5, -0.5); plt.gca().set_aspect("equal")
plt.title("2: anchor assignment by IoU"); plt.legend(fontsize=8); plt.show()

▶ What you'll see: two larger anchors cover the target equally well, and the first tied maximum is selected.

*Why it's done this way:* Dense detectors cannot let every nearby anchor learn the same object without creating duplicate responsibility. IoU gives a geometry-based assignment rule: the positive anchor is the one whose existing footprint needs the smallest correction. Ties remind us that label assignment is an algorithmic convention, not a mathematical law.

### 3. Non-maximum suppression removes duplicate echoes

A detector often emits several boxes for the same object. **NMS** sorts boxes by confidence, keeps the strongest remaining box, and suppresses lower-scored boxes whose IoU with it exceeds a threshold. The key is the sort: suppression only makes sense when the box doing the deleting is more trusted.

In [ ]:
boxes_w = np.array([[0., 0., 3., 3.],
                    [0.5, 0.5, 3.5, 3.5],
                    [5., 5., 7., 7.]])  # two overlapping boxes plus a far box.
scores_w = np.array([0.9, 0.8, 0.7])  # confidence scores.
thresh_w = 0.3  # duplicate threshold.
order_w = np.argsort(scores_w)[::-1]
print("score order:", order_w)
assert np.array_equal(order_w, [0, 1, 2])

▶ What you'll see: the 0.9 box is processed before the 0.8 and 0.7 boxes.

In [ ]:
iou_01_w = iou_one_w(boxes_w[0], boxes_w[1])
iou_02_w = iou_one_w(boxes_w[0], boxes_w[2])
print("IoU(box0, box1):", round(iou_01_w, 3))
print("IoU(box0, box2):", round(iou_02_w, 3))
assert round(iou_01_w, 3) == 0.532
assert iou_02_w == 0.0

▶ What you'll see: box 1 is a duplicate of box 0 at IoU 0.532, while box 2 is separate.

In [ ]:
def nms_w(boxes_w, scores_w, threshold_w):
    remaining_w = list(np.argsort(scores_w)[::-1])
    kept_w = []
    while remaining_w:
        current_w = remaining_w.pop(0)
        kept_w.append(current_w)
        remaining_w = [j_w for j_w in remaining_w if iou_one_w(boxes_w[current_w], boxes_w[j_w]) <= threshold_w]
    return kept_w

kept_w = nms_w(boxes_w, scores_w, thresh_w)
print("kept indices:", kept_w)
assert kept_w == [0, 2]

▶ What you'll see: box 1 is suppressed, so the final detections are `[0, 2]`.

In [ ]:
plt.figure(figsize=(5, 4))
for k_w, b_w in enumerate(boxes_w):
    color_w = "seagreen" if k_w in kept_w else "crimson"
    style_w = "-" if k_w in kept_w else "--"
    plt.gca().add_patch(plt.Rectangle((b_w[0], b_w[1]), b_w[2]-b_w[0], b_w[3]-b_w[1], fill=False, lw=3, ls=style_w, ec=color_w))
    plt.text(b_w[0], b_w[1]-0.15, f"{k_w}: {scores_w[k_w]:.1f}", color=color_w)
plt.xlim(-0.5, 7.5); plt.ylim(7.5, -0.5); plt.gca().set_aspect("equal")
plt.title("3: NMS keeps strong separated boxes"); plt.show()

▶ What you'll see: kept boxes are solid green, while the lower-scored duplicate is dashed red.

*Why it's done this way:* NMS implements “one confident box per object” without solving a global optimization problem. Greedy sorting is fast and matches the detector's confidence ranking; the IoU threshold encodes how much overlap still counts as the same object. Lower thresholds are aggressive, and higher thresholds allow more crowded or duplicate boxes to survive.

### 4. Precision-recall, AP, and mAP

A detector does not output just one score cutoff; it outputs a ranked list. As we scan from high confidence to low confidence, true positives increase recall and false positives reduce precision. **Average precision (AP)** integrates precision over the recall jumps. **mAP** averages AP across classes and often across IoU thresholds.

In [ ]:
recall_w = np.array([0.33, 0.67, 1.00])  # recall after each true positive jump.
precision_w = np.array([1.00, 0.75, 0.60])  # precision at those recall levels.
print("recall:", recall_w)
print("precision:", precision_w)

▶ What you'll see: precision decreases as more detections are admitted, while recall increases.

In [ ]:
recall_steps_w = np.r_[recall_w[0], np.diff(recall_w)]  # widths of PR rectangles.
ap_terms_w = precision_w * recall_steps_w  # rectangle areas under the PR curve.
ap_w = float(np.sum(ap_terms_w))
print("recall increments:", np.round(recall_steps_w, 2))
print("AP terms:", np.round(ap_terms_w, 3))
print("AP:", round(ap_w, 3))
assert round(ap_w, 3) == 0.783

▶ What you'll see: `0.33 + 0.255 + 0.198 = 0.783`, matching the source lesson.

In [ ]:
class_ap_w = np.array([0.783, 0.620, 0.910])  # AP for car, dog, bicycle in a toy evaluation.
map_w = float(np.mean(class_ap_w))
print("class APs:", class_ap_w)
print("mAP:", round(map_w, 3))
assert round(map_w, 3) == 0.771

▶ What you'll see: mAP is just the mean of class-level AP values in this toy setting.

In [ ]:
plt.figure(figsize=(4.5, 3.2))
plt.step(np.r_[0, recall_w], np.r_[precision_w[0], precision_w], where="post", color="purple")
for x0_w, dx_w, p_w in zip(np.r_[0, recall_w[:-1]], recall_steps_w, precision_w):
    plt.gca().add_patch(plt.Rectangle((x0_w, 0), dx_w, p_w, color="purple", alpha=0.15))
plt.xlabel("recall"); plt.ylabel("precision"); plt.ylim(0, 1.05); plt.xlim(0, 1.02)
plt.title(f"4: AP area = {ap_w:.3f}"); plt.show()

▶ What you'll see: AP is the purple area under the stepwise precision-recall curve.

*Why it's done this way:* Accuracy at one threshold would ignore ranking quality. AP rewards detectors that put correct, localized boxes early, because early high precision covers large recall jumps. Averaging AP into mAP prevents one easy class from hiding poor ranking on other classes, and changing the IoU threshold changes how strict “correct localization” is.

### 5. One-to-one matching and IoU thresholds

Detection evaluation needs one prediction to match one ground-truth object. Without that rule, duplicate boxes could all claim credit for the same object. A cost matrix makes the same idea explicit: choose the assignment with the smallest total cost, often combining class confidence and localization error.

In [ ]:
cost_w = np.array([[0.3, 1.1],
                   [0.7, 0.5]])  # prediction rows, target columns.
diag_cost_w = cost_w[0, 0] + cost_w[1, 1]
cross_cost_w = cost_w[0, 1] + cost_w[1, 0]
print("diagonal assignment cost:", diag_cost_w)
print("cross assignment cost:", cross_cost_w)
assert diag_cost_w == 0.8 and cross_cost_w == 1.8

▶ What you'll see: matching prediction 0→target 0 and prediction 1→target 1 is cheaper.

In [ ]:
pred_boxes_w = np.array([[0., 0., 3., 3.], [1., 1., 4., 4.]])
gt_boxes_w = np.array([[0., 0., 3., 3.], [5., 5., 7., 7.]])
iou_matrix_w = np.array([[iou_one_w(pbox_w, gbox_w) for gbox_w in gt_boxes_w] for pbox_w in pred_boxes_w])
print("IoU matrix:\n", np.round(iou_matrix_w, 3))
assert np.allclose(np.round(iou_matrix_w, 3), [[1.000, 0.000], [0.286, 0.000]])

▶ What you'll see: both predictions overlap the first object more than the second, so one-to-one matching is needed.

In [ ]:
thresholds_w = np.array([0.5, 0.75])
matched_iou_w = iou_matrix_w[0, 0]  # best one-to-one match for pred0 and gt0.
passes_w = matched_iou_w >= thresholds_w
print("matched IoU:", matched_iou_w)
print("passes thresholds 0.50 and 0.75:", passes_w)
assert np.array_equal(passes_w, [True, True])

▶ What you'll see: a perfect box passes both loose and strict localization thresholds.

In [ ]:
loose_iou_w = iou_matrix_w[1, 0]
print("shifted duplicate IoU with target 0:", round(loose_iou_w, 3))
print("passes IoU≥0.50?", bool(loose_iou_w >= 0.5))
assert round(loose_iou_w, 3) == 0.286
assert not bool(loose_iou_w >= 0.5)

▶ What you'll see: a centered-looking but shifted box fails even the 0.50 threshold.

*Why it's done this way:* Matching turns a cloud of predictions into counted true positives, false positives, and false negatives. The IoU threshold defines localization strictness, while one-to-one assignment prevents duplicate credit. That is why AP numbers must be compared only when the same class set, matching rule, and IoU threshold policy are being used.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for box arrays, vectorized IoU, sorting, and cumulative metrics.
import matplotlib.pyplot as plt  # load Matplotlib for box drawings and precision-recall curves.
np.random.seed(0)  # make random examples reproducible.

def box_area(box):  # compute area for a single [x1,y1,x2,y2] box.
    box = np.asarray(box, dtype=float)  # convert input so coordinate arithmetic is predictable.
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])  # clamp invalid widths/heights to zero.

def iou_box(box1, box2):  # compute IoU for two axis-aligned boxes.
    box1 = np.asarray(box1, dtype=float)  # ensure numeric array input.
    box2 = np.asarray(box2, dtype=float)  # ensure numeric array input.
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])  # intersection top-left corner.
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])  # intersection bottom-right corner.
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)  # intersection area, or 0 for no overlap.
    union = box_area(box1) + box_area(box2) - inter  # union area by inclusion-exclusion.
    return 0.0 if union == 0 else float(inter / union)  # guard empty boxes and return IoU.

def iou_matrix(boxes1, boxes2):  # compute all pairwise IoUs between two box sets.
    boxes1 = np.asarray(boxes1, dtype=float)  # rows are boxes in the first set.
    boxes2 = np.asarray(boxes2, dtype=float)  # rows are boxes in the second set.
    return np.array([[iou_box(a, b) for b in boxes2] for a in boxes1])  # small clear implementation.

def draw_boxes(boxes, labels=None, colors=None, title="boxes"):  # draw a small set of boxes for inspection.
    boxes = np.asarray(boxes, dtype=float)  # convert to array for indexing.
    labels = [str(i) for i in range(len(boxes))] if labels is None else labels  # default labels.
    colors = ["steelblue"] * len(boxes) if colors is None else colors  # default colors.
    plt.figure(figsize=(4.5, 4))  # compact square figure.
    for box, label, color in zip(boxes, labels, colors):  # draw each rectangle.
        plt.gca().add_patch(plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, ec=color, lw=2))  # add rectangle outline.
        plt.text(box[0], box[1] - 0.1, label, color=color)  # place label near the top-left corner.
    plt.xlim(-0.5, max(1.0, float(np.max(boxes[:, [0, 2]]))) + 0.8)  # set x-range from coordinates.
    plt.ylim(max(1.0, float(np.max(boxes[:, [1, 3]]))) + 0.8, -0.5)  # invert y-axis like image coordinates.
    plt.gca().set_aspect("equal")  # keep geometry honest.
    plt.title(title)  # add the supplied title.
    plt.show()  # render the figure.

def nms(boxes, scores, threshold):  # greedy non-maximum suppression.
    boxes = np.asarray(boxes, dtype=float)  # ensure box array.
    scores = np.asarray(scores, dtype=float)  # ensure score array.
    remaining = list(np.argsort(scores)[::-1])  # process highest confidence first.
    kept = []  # store surviving indices.
    while remaining:  # continue until every candidate is kept or suppressed.
        current = remaining.pop(0)  # choose the strongest remaining box.
        kept.append(current)  # keep it as the representative for this object region.
        remaining = [j for j in remaining if iou_box(boxes[current], boxes[j]) <= threshold]  # suppress high-IoU duplicates.
    return kept  # return original indices in keep order.

def ap_from_pr(recall, precision):  # AP as precision weighted by recall increments.
    recall = np.asarray(recall, dtype=float)  # ensure numeric recall.
    precision = np.asarray(precision, dtype=float)  # ensure numeric precision.
    dr = np.r_[recall[0], np.diff(recall)]  # recall step widths.
    return float(np.sum(precision * dr))  # rectangle area under the PR step curve.

## 🟢 Basics (warm-up)

### Basic 1 — Draw one box and compute its area

**Goal.** Convert corner coordinates into width, height, and area, because every later IoU number depends on this geometry. We build it in 2 steps.

In [ ]:
box_b1 = np.array([1., 2., 5., 6.])  # define one [x1,y1,x2,y2] box.
width_b1 = box_b1[2] - box_b1[0]  # compute horizontal span.
height_b1 = box_b1[3] - box_b1[1]  # compute vertical span.
print("width:", width_b1, "height:", height_b1)
assert width_b1 == 4.0 and height_b1 == 4.0

▶ What you'll see: the box is 4 units wide and 4 units tall.

In [ ]:
area_b1 = box_area(box_b1)  # compute area with the setup helper.
print("area:", area_b1)
assert area_b1 == 16.0
draw_boxes(np.array([box_b1]), labels=["area=16"], colors=["seagreen"], title="Basic 1: one detection box")

▶ What you'll see: one square box whose area is width times height.

👀 Takeaway: corner coordinates make box area simple arithmetic: `(x2-x1)(y2-y1)`.

### Basic 2 — Find an intersection rectangle

**Goal.** Compute the shared rectangle between two boxes, because IoU begins with overlap area. We build it in 2 steps.

In [ ]:
A_b2 = np.array([0., 0., 3., 3.])  # first box.
B_b2 = np.array([1., 1., 4., 4.])  # shifted box.
inter_b2 = np.array([max(A_b2[0], B_b2[0]), max(A_b2[1], B_b2[1]), min(A_b2[2], B_b2[2]), min(A_b2[3], B_b2[3])])  # intersection corners.
print("intersection box:", inter_b2)
assert np.array_equal(inter_b2, np.array([1., 1., 3., 3.]))

▶ What you'll see: the overlap box uses the inside-most left/top and right/bottom edges.

In [ ]:
inter_area_b2 = box_area(inter_b2)  # area of the intersection rectangle.
print("intersection area:", inter_area_b2)
assert inter_area_b2 == 4.0
draw_boxes(np.array([A_b2, B_b2, inter_b2]), labels=["A", "B", "A∩B"], colors=["steelblue", "orange", "green"], title="Basic 2: intersection rectangle")

▶ What you'll see: the green rectangle is the part both boxes cover.

👀 Takeaway: overlap is another rectangle, and its area is the numerator of IoU.

### Basic 3 — Compute IoU by hand

**Goal.** Divide intersection by union, because IoU should penalize both missing area and extra covered area. We build it in 3 steps.

In [ ]:
A_b3 = np.array([0., 0., 3., 3.])  # canonical source-lesson box A.
B_b3 = np.array([1., 1., 4., 4.])  # canonical source-lesson box B.
area_A_b3 = box_area(A_b3)  # area 9.
area_B_b3 = box_area(B_b3)  # area 9.
print("areas:", area_A_b3, area_B_b3)
assert area_A_b3 == 9.0 and area_B_b3 == 9.0

▶ What you'll see: both boxes cover 9 square units.

In [ ]:
inter_area_b3 = box_area(np.array([1., 1., 3., 3.]))  # shared 2-by-2 square.
union_b3 = area_A_b3 + area_B_b3 - inter_area_b3  # inclusion-exclusion union.
print("intersection:", inter_area_b3, "union:", union_b3)
assert union_b3 == 14.0

▶ What you'll see: union is `9+9-4=14`, not `18`, because the intersection would be double-counted.

In [ ]:
iou_b3 = iou_box(A_b3, B_b3)  # compute the same ratio with the helper.
print("IoU:", round(iou_b3, 3))
assert round(iou_b3, 3) == 0.286
plt.figure(figsize=(4, 3)); plt.bar(["intersection", "union", "IoU"], [inter_area_b3, union_b3, iou_b3], color=["green", "gray", "purple"]); plt.title("Basic 3: IoU ingredients"); plt.show()

▶ What you'll see: IoU is small compared with area counts because it is a ratio.

👀 Takeaway: IoU is `|A∩B| / |A∪B|`, so oversized and shifted boxes are both penalized.

### Basic 4 — Handle boxes with no overlap

**Goal.** Clamp intersection dimensions at zero, because non-overlap should produce IoU 0 rather than negative area. We build it in 2 steps.

In [ ]:
A_b4 = np.array([0., 0., 2., 2.])  # left box.
B_b4 = np.array([3., 3., 5., 5.])  # separated box.
raw_w_b4 = min(A_b4[2], B_b4[2]) - max(A_b4[0], B_b4[0])  # raw intersection width before clamping.
raw_h_b4 = min(A_b4[3], B_b4[3]) - max(A_b4[1], B_b4[1])  # raw intersection height before clamping.
print("raw width/height:", raw_w_b4, raw_h_b4)
assert raw_w_b4 == -1.0 and raw_h_b4 == -1.0

▶ What you'll see: separated boxes have negative raw spans if we do not clamp.

In [ ]:
iou_b4 = iou_box(A_b4, B_b4)  # helper clamps negative spans to zero.
print("IoU:", iou_b4)
assert iou_b4 == 0.0
draw_boxes(np.array([A_b4, B_b4]), labels=["A", "B"], colors=["steelblue", "crimson"], title="Basic 4: no overlap")

▶ What you'll see: the boxes are apart, so overlap and IoU are zero.

👀 Takeaway: clamping at zero is what makes IoU robust for separated boxes.

### Basic 5 — Score anchors against one target

**Goal.** Compute anchor IoUs, because detectors assign a ground-truth object to the best-overlapping preset box. We build it in 3 steps.

In [ ]:
gt_b5 = np.array([1., 1., 3., 3.])  # ground-truth object.
anchors_b5 = np.array([[0., 0., 2., 2.], [0., 0., 3., 3.], [1., 1., 4., 4.]])  # candidate anchors.
print("number of anchors:", len(anchors_b5))
assert len(anchors_b5) == 3

▶ What you'll see: three possible anchors compete for one target.

In [ ]:
iou_scores_b5 = np.array([iou_box(anchor_b5, gt_b5) for anchor_b5 in anchors_b5])  # score each anchor.
print("anchor IoUs:", np.round(iou_scores_b5, 3))
assert np.allclose(np.round(iou_scores_b5, 3), [0.143, 0.444, 0.444])

▶ What you'll see: two anchors tie for best overlap with the object.

In [ ]:
best_b5 = int(np.argmax(iou_scores_b5))  # choose the first best anchor.
print("assigned anchor:", best_b5)
assert best_b5 == 1
plt.figure(figsize=(4, 3)); plt.bar(["a0", "a1", "a2"], iou_scores_b5, color=["gray", "seagreen", "orange"]); plt.title("Basic 5: anchor IoUs"); plt.ylabel("IoU with GT"); plt.show()

▶ What you'll see: anchors 1 and 2 have equal-height bars, but anchor 1 is selected first.

👀 Takeaway: anchor assignment is best-IoU matching plus a deterministic tie rule.

### Basic 6 — Sort detections by confidence

**Goal.** Rank predicted boxes before NMS and AP, because both procedures assume high-confidence boxes are considered first. We build it in 2 steps.

In [ ]:
scores_b6 = np.array([0.55, 0.90, 0.20, 0.75])  # detector confidences.
order_b6 = np.argsort(scores_b6)[::-1]  # descending score order.
print("sorted indices:", order_b6)
assert np.array_equal(order_b6, [1, 3, 0, 2])

▶ What you'll see: the highest confidence index comes first.

In [ ]:
sorted_scores_b6 = scores_b6[order_b6]  # reorder scores for inspection.
print("sorted scores:", sorted_scores_b6)
plt.figure(figsize=(4, 3)); plt.bar(np.arange(len(sorted_scores_b6)), sorted_scores_b6, color="teal"); plt.title("Basic 6: confidence ranking"); plt.xlabel("rank"); plt.ylabel("score"); plt.show()

▶ What you'll see: bars decrease from left to right after sorting.

👀 Takeaway: confidence ranking is the backbone of greedy NMS and precision-recall evaluation.

### Basic 7 — Suppress one duplicate box

**Goal.** Compare two overlapping detections against an NMS threshold, because duplicate removal is just an IoU decision after sorting. We build it in 2 steps.

In [ ]:
keep_box_b7 = np.array([0., 0., 3., 3.])  # stronger detection.
dup_box_b7 = np.array([0.5, 0.5, 3.5, 3.5])  # weaker nearby detection.
overlap_b7 = iou_box(keep_box_b7, dup_box_b7)  # duplicate overlap.
print("IoU with kept box:", round(overlap_b7, 3))
assert round(overlap_b7, 3) == 0.532

▶ What you'll see: the weaker box overlaps the stronger one by more than half the union.

In [ ]:
threshold_b7 = 0.3  # NMS duplicate threshold.
suppress_b7 = overlap_b7 > threshold_b7  # true means remove duplicate.
print("suppress duplicate?", bool(suppress_b7))
assert bool(suppress_b7)
draw_boxes(np.array([keep_box_b7, dup_box_b7]), labels=["keep", "suppress"], colors=["seagreen", "crimson"], title="Basic 7: duplicate suppression")

▶ What you'll see: two boxes cover the same object region, so the lower-scored one is suppressed.

👀 Takeaway: NMS removes a lower-ranked box when its IoU with a kept box is above the threshold.

### Basic 8 — Run tiny NMS

**Goal.** Execute the full greedy NMS loop on three boxes, because the kept set is not necessarily the top-k scores. We build it in 3 steps.

In [ ]:
boxes_b8 = np.array([[0., 0., 3., 3.], [0.5, 0.5, 3.5, 3.5], [5., 5., 7., 7.]])  # two duplicates plus one separate object.
scores_b8 = np.array([0.9, 0.8, 0.7])  # descending confidences already.
print("boxes:", boxes_b8.shape, "scores:", scores_b8)
assert boxes_b8.shape == (3, 4)

▶ What you'll see: three candidate detections are ready for cleanup.

In [ ]:
kept_b8 = nms(boxes_b8, scores_b8, 0.3)  # run setup NMS helper.
print("kept indices:", kept_b8)
assert kept_b8 == [0, 2]

▶ What you'll see: NMS keeps the strongest duplicate and the far-away box.

In [ ]:
colors_b8 = ["seagreen" if i in kept_b8 else "crimson" for i in range(len(boxes_b8))]  # color by NMS outcome.
draw_boxes(boxes_b8, labels=[f"{i}: {scores_b8[i]:.1f}" for i in range(3)], colors=colors_b8, title="Basic 8: NMS output")

▶ What you'll see: the duplicate with score 0.8 is red while detections 0 and 2 are green.

👀 Takeaway: NMS chooses the strongest box per sufficiently separated object region.

### Basic 9 — Compute precision and recall

**Goal.** Convert ranked true/false positives into precision and recall, because AP is built from these cumulative quantities. We build it in 3 steps.

In [ ]:
tp_flags_b9 = np.array([1, 1, 0, 1, 0])  # 1 means this ranked detection is a true positive.
num_gt_b9 = 3  # total ground-truth objects.
fp_flags_b9 = 1 - tp_flags_b9  # false positives are the opposite in this toy ranked list.
print("TP flags:", tp_flags_b9, "FP flags:", fp_flags_b9)
assert int(tp_flags_b9.sum()) == num_gt_b9

▶ What you'll see: five detections include three true positives and two false positives.

In [ ]:
cum_tp_b9 = np.cumsum(tp_flags_b9)  # true positives found up to each rank.
cum_fp_b9 = np.cumsum(fp_flags_b9)  # false positives accumulated up to each rank.
precision_b9 = cum_tp_b9 / (cum_tp_b9 + cum_fp_b9)  # fraction of admitted detections that are correct.
recall_b9 = cum_tp_b9 / num_gt_b9  # fraction of objects found.
print("precision:", np.round(precision_b9, 3))
print("recall:", np.round(recall_b9, 3))

▶ What you'll see: recall rises with true positives, while precision drops when false positives arrive.

In [ ]:
plt.figure(figsize=(4, 3)); plt.step(recall_b9, precision_b9, where="post", color="purple"); plt.ylim(0, 1.05); plt.xlim(0, 1.05); plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Basic 9: PR curve from ranked detections"); plt.show()

▶ What you'll see: a stepwise precision-recall curve generated by walking down the ranked list.

👀 Takeaway: precision measures ranking cleanliness; recall measures how much object coverage the ranking has reached.

### Basic 10 — Integrate AP and average mAP

**Goal.** Compute AP from recall jumps and mAP from class APs, because detection evaluation averages ranking quality across classes. We build it in 3 steps.

In [ ]:
recall_b10 = np.array([0.33, 0.67, 1.0])  # recall after each successful detection.
precision_b10 = np.array([1.0, 0.75, 0.6])  # precision at each recall step.
ap_b10 = ap_from_pr(recall_b10, precision_b10)  # integrate precision over recall increments.
print("AP:", round(ap_b10, 3))
assert round(ap_b10, 3) == 0.783

▶ What you'll see: AP is 0.783 for the toy ranked detector.

In [ ]:
aps_b10 = np.array([ap_b10, 0.62, 0.91])  # three class-level AP values.
map_b10 = float(np.mean(aps_b10))  # mean AP over classes.
print("mAP:", round(map_b10, 3))
assert round(map_b10, 3) == 0.771

▶ What you'll see: mAP summarizes several class AP scores with an ordinary mean.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["car", "dog", "bike", "mAP"], [aps_b10[0], aps_b10[1], aps_b10[2], map_b10], color=["steelblue", "orange", "green", "purple"]); plt.ylim(0, 1); plt.title("Basic 10: class APs and mAP"); plt.ylabel("score"); plt.show()

▶ What you'll see: mAP sits between the class AP values it averages.

👀 Takeaway: AP scores one class ranking; mAP averages AP so evaluation reflects all classes.

## 🟡 Easy

### Easy 1 — Vectorize pairwise IoU

**Goal.** Build an IoU matrix between predictions and targets, because matching algorithms need every possible prediction-target overlap. We build it in 3 steps.

In [ ]:
preds_e1 = np.array([[0., 0., 3., 3.], [1., 1., 4., 4.], [5., 5., 7., 7.]])  # predicted boxes.
targets_e1 = np.array([[0., 0., 3., 3.], [5., 5., 7., 7.]])  # ground-truth boxes.
print("predictions:", preds_e1.shape, "targets:", targets_e1.shape)
assert preds_e1.shape == (3, 4) and targets_e1.shape == (2, 4)

▶ What you'll see: three predictions will be compared with two targets.

In [ ]:
ious_e1 = iou_matrix(preds_e1, targets_e1)  # compute pairwise IoUs.
print("IoU matrix:\n", np.round(ious_e1, 3))
assert np.allclose(np.round(ious_e1, 3), [[1.000, 0.000], [0.286, 0.000], [0.000, 1.000]])

▶ What you'll see: each row tells how one prediction overlaps every target.

In [ ]:
best_target_e1 = np.argmax(ious_e1, axis=1)  # best target per prediction.
best_iou_e1 = np.max(ious_e1, axis=1)  # best overlap per prediction.
print("best targets:", best_target_e1)
print("best IoUs:", np.round(best_iou_e1, 3))
plt.figure(figsize=(4, 3)); plt.imshow(ious_e1, cmap="viridis", vmin=0, vmax=1); plt.colorbar(label="IoU"); plt.title("Easy 1: prediction-target IoU matrix"); plt.xlabel("target"); plt.ylabel("prediction"); plt.show()

▶ What you'll see: bright diagonal-ish cells reveal likely matches.

👀 Takeaway: an IoU matrix turns matching into a table lookup problem instead of pairwise guesswork.

### Easy 2 — Assign anchors with positive and negative labels

**Goal.** Label anchors as positive, negative, or ignored, because detector training needs responsibility targets rather than just raw IoU scores. We build it in 3 steps.

In [ ]:
gt_e2 = np.array([1., 1., 3., 3.])  # object box.
anchors_e2 = np.array([[0., 0., 2., 2.], [0., 0., 3., 3.], [1., 1., 4., 4.], [4., 4., 6., 6.]])  # candidates plus a background anchor.
ious_e2 = np.array([iou_box(anchor_e2, gt_e2) for anchor_e2 in anchors_e2])  # overlap with target.
print("IoUs:", np.round(ious_e2, 3))
assert np.allclose(np.round(ious_e2, 3), [0.143, 0.444, 0.444, 0.000])

▶ What you'll see: one anchor is clear background, and two tie for best object overlap.

In [ ]:
labels_e2 = np.full(len(anchors_e2), "ignore", dtype=object)  # start with ignored anchors.
labels_e2[ious_e2 < 0.2] = "negative"  # low IoU means background.
labels_e2[int(np.argmax(ious_e2))] = "positive"  # force the best anchor positive even if below a usual 0.5 threshold.
print("anchor labels:", labels_e2)
assert labels_e2.tolist() == ["negative", "positive", "ignore", "negative"]

▶ What you'll see: the first maximum gets the positive label; low-overlap anchors become negatives.

In [ ]:
colors_e2 = [{"positive":"seagreen", "negative":"crimson", "ignore":"gray"}[lab_e2] for lab_e2 in labels_e2]  # color by training label.
draw_boxes(anchors_e2, labels=[f"{i}:{labels_e2[i]}" for i in range(len(anchors_e2))], colors=colors_e2, title="Easy 2: anchor labels")

▶ What you'll see: one positive anchor, two negatives, and one ignored tied anchor.

👀 Takeaway: anchor labels combine threshold rules with a best-anchor guarantee so every object has a responsible predictor.

### Easy 3 — Run class-wise NMS

**Goal.** Apply NMS separately per class, because a dog box should not suppress a bicycle box just because they overlap. We build it in 4 steps.

In [ ]:
boxes_e3 = np.array([[0., 0., 3., 3.], [0.4, 0.4, 3.4, 3.4], [0.2, 0.2, 3.2, 3.2], [5., 5., 7., 7.]])  # overlapping and far boxes.
scores_e3 = np.array([0.90, 0.80, 0.70, 0.60])  # confidences.
classes_e3 = np.array([0, 0, 1, 0])  # class 1 overlaps class 0 but should be handled separately.
print("classes:", classes_e3)
assert np.array_equal(classes_e3, [0, 0, 1, 0])

▶ What you'll see: three class-0 boxes and one class-1 box.

In [ ]:
kept_e3 = []  # collect original indices kept across classes.
for cls_e3 in np.unique(classes_e3):  # run NMS within each class.
    idx_e3 = np.where(classes_e3 == cls_e3)[0]  # indices for this class.
    local_keep_e3 = nms(boxes_e3[idx_e3], scores_e3[idx_e3], 0.3)  # NMS on that class only.
    kept_e3.extend(idx_e3[local_keep_e3].tolist())  # map back to original indices.
kept_e3 = sorted(kept_e3)  # sort for readable output.
print("kept after class-wise NMS:", kept_e3)
assert kept_e3 == [0, 2, 3]

▶ What you'll see: class-1 box 2 survives even though it overlaps class-0 box 0.

In [ ]:
kept_global_e3 = nms(boxes_e3, scores_e3, 0.3)  # wrong comparison: class-agnostic NMS.
print("kept if class-agnostic:", kept_global_e3)
assert kept_global_e3 == [0, 3]

▶ What you'll see: global NMS would incorrectly delete the class-1 detection.

In [ ]:
colors_e3 = ["seagreen" if i in kept_e3 else "crimson" for i in range(len(boxes_e3))]  # class-wise outcome colors.
draw_boxes(boxes_e3, labels=[f"{i}:c{classes_e3[i]}" for i in range(len(boxes_e3))], colors=colors_e3, title="Easy 3: class-wise NMS")

▶ What you'll see: overlapping boxes of different classes can both remain green.

👀 Takeaway: NMS should usually be per class so geometric overlap does not erase semantically different objects.

### Easy 4 — Match detections and compute a PR curve

**Goal.** Turn scored detections into true/false positives with greedy one-to-one matching, because AP needs ranked correctness flags. We build it in 4 steps.

In [ ]:
gt_e4 = np.array([[0., 0., 3., 3.], [5., 5., 7., 7.]])  # two objects.
dets_e4 = np.array([[0., 0., 3., 3.], [0.5, 0.5, 3.5, 3.5], [5., 5., 7., 7.], [8., 8., 9., 9.]])  # predictions.
scores_e4 = np.array([0.95, 0.90, 0.70, 0.60])  # ranked scores.
order_e4 = np.argsort(scores_e4)[::-1]  # evaluate high confidence first.
print("evaluation order:", order_e4)
assert np.array_equal(order_e4, [0, 1, 2, 3])

▶ What you'll see: detections are already sorted by decreasing confidence.

In [ ]:
matched_e4 = np.zeros(len(gt_e4), dtype=bool)  # whether each target is already claimed.
tp_e4 = []  # true-positive flags.
for idx_e4 in order_e4:  # scan detections by confidence.
    ious_now_e4 = np.array([iou_box(dets_e4[idx_e4], g_e4) for g_e4 in gt_e4])  # overlaps with all targets.
    best_e4 = int(np.argmax(ious_now_e4))  # best target.
    is_tp_e4 = (ious_now_e4[best_e4] >= 0.5) and (not matched_e4[best_e4])  # IoU threshold plus one-to-one rule.
    tp_e4.append(1 if is_tp_e4 else 0)  # record correctness.
    if is_tp_e4:
        matched_e4[best_e4] = True  # claim the target.
print("TP flags:", tp_e4)
assert tp_e4 == [1, 0, 1, 0]

▶ What you'll see: the duplicate of the first object is a false positive because that target is already matched.

In [ ]:
tp_e4 = np.array(tp_e4)  # convert flags to array.
fp_e4 = 1 - tp_e4  # false positives.
precision_e4 = np.cumsum(tp_e4) / (np.cumsum(tp_e4) + np.cumsum(fp_e4))  # cumulative precision.
recall_e4 = np.cumsum(tp_e4) / len(gt_e4)  # cumulative recall.
print("precision:", np.round(precision_e4, 3))
print("recall:", np.round(recall_e4, 3))
assert np.allclose(np.round(precision_e4, 3), [1.000, 0.500, 0.667, 0.500])

▶ What you'll see: the duplicate hurts precision without increasing recall.

In [ ]:
plt.figure(figsize=(4, 3)); plt.step(recall_e4, precision_e4, where="post", color="purple"); plt.ylim(0, 1.05); plt.xlim(0, 1.05); plt.title("Easy 4: PR after matching"); plt.xlabel("recall"); plt.ylabel("precision"); plt.show()

▶ What you'll see: precision dips at false positives and recall only jumps on new matched objects.

👀 Takeaway: one-to-one matching is what makes duplicate detections count as false positives.

### Easy 5 — Compare AP at two IoU thresholds

**Goal.** Evaluate the same ranked detections at IoU 0.50 and 0.75, because stricter localization thresholds can lower AP without changing confidence scores. We build it in 4 steps.

In [ ]:
gt_e5 = np.array([[0., 0., 3., 3.], [5., 5., 7., 7.]])  # two objects.
dets_e5 = np.array([[0., 0., 3., 3.], [5.2, 5.2, 7.2, 7.2], [0.5, 0.5, 3.5, 3.5]])  # good, shifted, duplicate.
scores_e5 = np.array([0.95, 0.85, 0.70])  # confidence ranking.
print("candidate IoUs to best target:", np.round(np.max(iou_matrix(dets_e5, gt_e5), axis=1), 3))
assert np.allclose(np.round(np.max(iou_matrix(dets_e5, gt_e5), axis=1), 3), [1.000, 0.681, 0.532])

▶ What you'll see: the shifted second object is good at 0.50 but not strict enough for 0.75.

In [ ]:
def pr_for_threshold_e5(threshold_e5):
    matched_e5 = np.zeros(len(gt_e5), dtype=bool)
    tp_e5 = []
    for idx_e5 in np.argsort(scores_e5)[::-1]:
        overlaps_e5 = np.array([iou_box(dets_e5[idx_e5], g_e5) for g_e5 in gt_e5])
        best_e5 = int(np.argmax(overlaps_e5))
        ok_e5 = (overlaps_e5[best_e5] >= threshold_e5) and (not matched_e5[best_e5])
        tp_e5.append(1 if ok_e5 else 0)
        if ok_e5:
            matched_e5[best_e5] = True
    tp_e5 = np.array(tp_e5); fp_e5 = 1 - tp_e5
    return np.cumsum(tp_e5) / len(gt_e5), np.cumsum(tp_e5) / (np.cumsum(tp_e5) + np.cumsum(fp_e5))

rec50_e5, prec50_e5 = pr_for_threshold_e5(0.50)
rec75_e5, prec75_e5 = pr_for_threshold_e5(0.75)
print("recall @0.50:", rec50_e5, "precision @0.50:", np.round(prec50_e5, 3))

▶ What you'll see: at IoU 0.50, the first two detections can recover both objects.

In [ ]:
ap50_e5 = ap_from_pr(rec50_e5, prec50_e5)  # AP at loose localization.
ap75_e5 = ap_from_pr(rec75_e5, prec75_e5)  # AP at strict localization.
print("AP50:", round(ap50_e5, 3), "AP75:", round(ap75_e5, 3))
assert round(ap50_e5, 3) == 1.0
assert round(ap75_e5, 3) == 0.5

▶ What you'll see: strict IoU lowers AP because the shifted object no longer counts.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["AP@0.50", "AP@0.75"], [ap50_e5, ap75_e5], color=["seagreen", "crimson"]); plt.ylim(0, 1.05); plt.title("Easy 5: localization strictness changes AP"); plt.ylabel("AP"); plt.show()

▶ What you'll see: the same detections score higher under the looser IoU threshold.

👀 Takeaway: AP numbers are comparable only when the IoU threshold policy is the same.

## 🔴 Advanced

### Advanced 1 — Sweep anchor sizes for best coverage

**Goal.** Compare anchor templates against several targets, because good anchors give high best-IoU coverage before regression learns corrections. We build it in 4 steps.

In [ ]:
targets_a1 = np.array([[0., 0., 2., 2.], [0., 0., 4., 2.], [0., 0., 2., 5.]])  # square, wide, tall objects at same origin.
anchor_sizes_a1 = np.array([[2., 2.], [4., 2.], [2., 4.], [5., 5.]])  # candidate width-height templates.
anchors_a1 = np.c_[np.zeros(len(anchor_sizes_a1)), np.zeros(len(anchor_sizes_a1)), anchor_sizes_a1]  # convert sizes to corner boxes.
print("anchors:\n", anchors_a1)
assert anchors_a1.shape == (4, 4)

▶ What you'll see: anchors are centered at the origin for a pure shape-coverage demo.

In [ ]:
ious_a1 = iou_matrix(targets_a1, anchors_a1)  # target-by-anchor overlap table.
best_iou_a1 = np.max(ious_a1, axis=1)  # best anchor per target.
best_anchor_a1 = np.argmax(ious_a1, axis=1)  # responsible anchor index per target.
print("best anchors:", best_anchor_a1)
print("best IoUs:", np.round(best_iou_a1, 3))
assert np.allclose(np.round(best_iou_a1, 3), [1.0, 1.0, 0.8])

▶ What you'll see: square and wide targets have perfect templates, while the tall target is approximated.

In [ ]:
coverage_a1 = float(np.mean(best_iou_a1 >= 0.5))  # fraction of targets covered by at least one good anchor.
mean_best_a1 = float(np.mean(best_iou_a1))  # average best overlap.
print("coverage@0.5:", coverage_a1, "mean best IoU:", round(mean_best_a1, 3))
assert coverage_a1 == 1.0

▶ What you'll see: every target has a usable anchor at the 0.5 threshold.

In [ ]:
plt.figure(figsize=(5, 3)); plt.imshow(ious_a1, cmap="viridis", vmin=0, vmax=1); plt.colorbar(label="IoU"); plt.title("Advanced 1: target-anchor IoUs"); plt.xlabel("anchor"); plt.ylabel("target"); plt.show()

▶ What you'll see: the brightest cell in each row is the anchor that receives responsibility.

👀 Takeaway: anchor design is a coverage problem: high best-IoU anchors make regression targets easier.

### Advanced 2 — Soft-NMS decays duplicate scores

**Goal.** Compare hard NMS with score decay, because Soft-NMS reduces duplicate confidence instead of deleting boxes immediately. We build it in 4 steps.

In [ ]:
boxes_a2 = np.array([[0., 0., 3., 3.], [0.5, 0.5, 3.5, 3.5], [5., 5., 7., 7.]])  # duplicate pair plus separate box.
scores_a2 = np.array([0.90, 0.80, 0.70])  # original scores.
print("original scores:", scores_a2)
assert np.allclose(scores_a2, [0.9, 0.8, 0.7])

▶ What you'll see: box 1 starts as a strong duplicate.

In [ ]:
hard_keep_a2 = nms(boxes_a2, scores_a2, 0.3)  # hard NMS removes duplicate box 1.
print("hard NMS kept:", hard_keep_a2)
assert hard_keep_a2 == [0, 2]

▶ What you'll see: hard NMS deletes the high-overlap duplicate.

In [ ]:
soft_scores_a2 = scores_a2.copy()  # keep a mutable score vector.
first_a2 = int(np.argmax(soft_scores_a2))  # highest score box.
for j_a2 in range(len(boxes_a2)):
    if j_a2 != first_a2:
        soft_scores_a2[j_a2] *= (1 - iou_box(boxes_a2[first_a2], boxes_a2[j_a2]))  # linear decay by overlap.
print("soft scores after first keep:", np.round(soft_scores_a2, 3))
assert np.allclose(np.round(soft_scores_a2, 3), [0.900, 0.374, 0.700])

▶ What you'll see: the duplicate score is decayed from 0.8 to about 0.374, while the far box stays 0.7.

In [ ]:
plt.figure(figsize=(5, 3)); x_a2 = np.arange(3); plt.bar(x_a2 - 0.18, scores_a2, width=0.36, label="original", color="gray"); plt.bar(x_a2 + 0.18, soft_scores_a2, width=0.36, label="soft-decayed", color="purple"); plt.xticks(x_a2, ["box0", "box1", "box2"]); plt.title("Advanced 2: Soft-NMS score decay"); plt.legend(); plt.show()

▶ What you'll see: overlapping duplicates lose rank without being instantly removed.

👀 Takeaway: Soft-NMS keeps the greedy spirit but changes suppression from a hard threshold to a confidence penalty.

### Advanced 3 — Build AP from raw detections end to end

**Goal.** Compute AP directly from boxes, scores, and ground truth, because a detector metric is matching plus cumulative ranking plus integration. We build it in 5 steps.

In [ ]:
gt_a3 = np.array([[0., 0., 3., 3.], [5., 5., 7., 7.], [8., 1., 10., 3.]])  # three objects.
dets_a3 = np.array([[0., 0., 3., 3.], [5.2, 5.2, 7.2, 7.2], [0.5, 0.5, 3.5, 3.5], [8., 1., 10., 3.], [12., 12., 13., 13.]])  # five detections.
scores_a3 = np.array([0.95, 0.90, 0.80, 0.70, 0.40])  # confidence scores.
print("detections:", len(dets_a3), "ground truths:", len(gt_a3))
assert len(dets_a3) == 5 and len(gt_a3) == 3

▶ What you'll see: five ranked detections compete for three target objects.

In [ ]:
order_a3 = np.argsort(scores_a3)[::-1]  # high-to-low evaluation order.
matched_a3 = np.zeros(len(gt_a3), dtype=bool)  # target claim flags.
tp_a3 = []  # true-positive flags in ranked order.
for idx_a3 in order_a3:
    overlaps_a3 = np.array([iou_box(dets_a3[idx_a3], g_a3) for g_a3 in gt_a3])
    best_a3 = int(np.argmax(overlaps_a3))
    ok_a3 = overlaps_a3[best_a3] >= 0.5 and not matched_a3[best_a3]
    tp_a3.append(1 if ok_a3 else 0)
    if ok_a3:
        matched_a3[best_a3] = True
print("TP flags:", tp_a3)
assert tp_a3 == [1, 1, 0, 1, 0]

▶ What you'll see: the duplicate and background detections are false positives.

In [ ]:
tp_a3 = np.array(tp_a3)  # true-positive array.
fp_a3 = 1 - tp_a3  # false-positive array.
precision_a3 = np.cumsum(tp_a3) / (np.cumsum(tp_a3) + np.cumsum(fp_a3))  # ranked precision.
recall_a3 = np.cumsum(tp_a3) / len(gt_a3)  # ranked recall.
print("precision:", np.round(precision_a3, 3))
print("recall:", np.round(recall_a3, 3))
assert np.allclose(np.round(recall_a3, 3), [0.333, 0.667, 0.667, 1.000, 1.000])

▶ What you'll see: false positives leave recall unchanged while lowering precision.

In [ ]:
recall_jumps_a3 = np.r_[recall_a3[0], np.diff(recall_a3)]  # recall increments, zero for false positives.
ap_a3 = float(np.sum(precision_a3 * recall_jumps_a3))  # AP from every rank.
print("AP:", round(ap_a3, 3))
assert round(ap_a3, 3) == 0.917

▶ What you'll see: AP is high because the first two detections are true positives.

In [ ]:
plt.figure(figsize=(4, 3)); plt.step(recall_a3, precision_a3, where="post", color="navy"); plt.ylim(0, 1.05); plt.xlim(0, 1.05); plt.title(f"Advanced 3: end-to-end AP = {ap_a3:.3f}"); plt.xlabel("recall"); plt.ylabel("precision"); plt.show()

▶ What you'll see: one mid-rank false positive dents precision before full recall is reached.

👀 Takeaway: AP is not a separate trick; it is the area produced by matching the ranked detections one by one.

### Advanced 4 — Compute mAP across classes and IoU thresholds

**Goal.** Average AP over classes and thresholds, because modern detection reports often summarize both semantic and localization difficulty. We build it in 4 steps.

In [ ]:
aps_by_class_threshold_a4 = np.array([[0.95, 0.70],
                                      [0.80, 0.55],
                                      [0.60, 0.30]])  # rows=classes, cols=IoU thresholds 0.50 and 0.75.
print("AP table:\n", aps_by_class_threshold_a4)
assert aps_by_class_threshold_a4.shape == (3, 2)

▶ What you'll see: every class has lower AP at the stricter threshold.

In [ ]:
map_by_threshold_a4 = np.mean(aps_by_class_threshold_a4, axis=0)  # average over classes at each threshold.
map_by_class_a4 = np.mean(aps_by_class_threshold_a4, axis=1)  # average over thresholds for each class.
overall_map_a4 = float(np.mean(aps_by_class_threshold_a4))  # average over all entries.
print("mAP by threshold:", np.round(map_by_threshold_a4, 3))
print("class means:", np.round(map_by_class_a4, 3))
assert np.allclose(np.round(map_by_threshold_a4, 3), [0.783, 0.517])

▶ What you'll see: localization strictness lowers the threshold-level average.

In [ ]:
print("overall mAP:", round(overall_map_a4, 3))
assert round(overall_map_a4, 3) == 0.65

▶ What you'll see: averaging all class-threshold APs gives 0.650.

In [ ]:
plt.figure(figsize=(5, 3)); plt.imshow(aps_by_class_threshold_a4, cmap="viridis", vmin=0, vmax=1); plt.colorbar(label="AP"); plt.xticks([0, 1], ["IoU .50", "IoU .75"]); plt.yticks([0, 1, 2], ["car", "dog", "bike"]); plt.title(f"Advanced 4: overall mAP = {overall_map_a4:.3f}"); plt.show()

▶ What you'll see: bright cells mark easier class-threshold combinations; the strict column is darker.

👀 Takeaway: mAP is an averaging convention, so always report which classes and IoU thresholds were included.

### Advanced 5 — Show how NMS threshold changes AP

**Goal.** Sweep NMS thresholds before evaluation, because suppression settings can trade duplicate false positives against missed crowded objects. We build it in 5 steps.

In [ ]:
gt_a5 = np.array([[0., 0., 3., 3.], [3.2, 0., 6.2, 3.]])  # two nearby objects.
boxes_a5 = np.array([[0., 0., 3., 3.], [0.3, 0., 3.3, 3.], [3.2, 0., 6.2, 3.]])  # first object, duplicate, second object.
scores_a5 = np.array([0.95, 0.90, 0.85])  # confidence ranking.
print("IoU between true object boxes:", round(iou_box(gt_a5[0], gt_a5[1]), 3))
assert round(iou_box(gt_a5[0], gt_a5[1]), 3) == 0.0

▶ What you'll see: the true objects are adjacent, but the duplicate overlaps both regions enough to matter.

In [ ]:
def eval_after_nms_a5(nms_thresh_a5):
    kept_a5 = nms(boxes_a5, scores_a5, nms_thresh_a5)
    kept_order_a5 = sorted(kept_a5, key=lambda i_a5: -scores_a5[i_a5])
    matched_a5 = np.zeros(len(gt_a5), dtype=bool)
    tp_a5 = []
    for idx_a5 in kept_order_a5:
        overlaps_a5 = np.array([iou_box(boxes_a5[idx_a5], g_a5) for g_a5 in gt_a5])
        best_a5 = int(np.argmax(overlaps_a5))
        ok_a5 = overlaps_a5[best_a5] >= 0.5 and not matched_a5[best_a5]
        tp_a5.append(1 if ok_a5 else 0)
        if ok_a5:
            matched_a5[best_a5] = True
    if len(tp_a5) == 0:
        return kept_a5, 0.0
    tp_a5 = np.array(tp_a5); fp_a5 = 1 - tp_a5
    prec_a5 = np.cumsum(tp_a5) / (np.cumsum(tp_a5) + np.cumsum(fp_a5))
    rec_a5 = np.cumsum(tp_a5) / len(gt_a5)
    return kept_a5, ap_from_pr(rec_a5, prec_a5)

thresholds_a5 = np.array([0.1, 0.3, 0.7])  # aggressive to lenient suppression.
print("thresholds:", thresholds_a5)
assert len(thresholds_a5) == 3

▶ What you'll see: the sweep will compare three NMS policies.

In [ ]:
kept_lists_a5 = []  # keep outcomes.
ap_values_a5 = []  # AP outcomes.
for t_a5 in thresholds_a5:
    kept_a5, ap_val_a5 = eval_after_nms_a5(float(t_a5))
    kept_lists_a5.append(kept_a5)
    ap_values_a5.append(ap_val_a5)
print("kept lists:", kept_lists_a5)
print("AP values:", np.round(ap_values_a5, 3))
assert kept_lists_a5[0] == [0, 2]

▶ What you'll see: low and moderate NMS remove the duplicate, while high NMS may leave it.

In [ ]:
best_idx_a5 = int(np.argmax(ap_values_a5))  # choose the best AP setting in this toy sweep.
best_thresh_a5 = float(thresholds_a5[best_idx_a5])
print("best NMS threshold:", best_thresh_a5, "AP:", round(ap_values_a5[best_idx_a5], 3))
assert round(max(ap_values_a5), 3) == 1.0

▶ What you'll see: at least one threshold reaches perfect AP on this small example.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(thresholds_a5, ap_values_a5, marker="o", color="crimson"); plt.ylim(0, 1.05); plt.xlabel("NMS IoU threshold"); plt.ylabel("AP"); plt.title("Advanced 5: NMS threshold affects metric"); plt.show()

▶ What you'll see: AP changes as suppression becomes more or less aggressive.

👀 Takeaway: NMS is not just post-processing; its threshold can change measured detector quality.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Detection metrics are the contract that turns messy boxes into decisions: overlap decides matches, NMS removes echoes, and AP rewards confident recall.

Object detectors emit many rectangles with scores. Coordinate geometry, greedy NMS, and precision-recall bookkeeping turn that cloud into fair decisions.

Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
rng = np.random.default_rng(7)

## Build IoU, anchor assignment, NMS, and AP

For boxes $A$ and $B$, $$\operatorname{IoU}(A,B)=\frac{|A\cap B|}{|A|+|B|-|A\cap B|}.$$ The D1 lesson boxes prove the arithmetic before we scale up.

In [ ]:
def box_area(box):
    w = max(0.0, float(box[2] - box[0]))
    h = max(0.0, float(box[3] - box[1]))
    return w * h


def iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = box_area([x1, y1, x2, y2])
    union = box_area(a) + box_area(b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def nms(boxes, scores, threshold=0.3, sort=True):
    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float)
    if sort:
        order = list(np.argsort(-scores))
    else:
        order = list(range(len(scores)))
    keep = []
    while order:
        current = order.pop(0)
        keep.append(int(current))
        rest = []
        for idx in order:
            overlap = iou(boxes[current], boxes[idx])
            if overlap <= threshold:
                rest.append(idx)
        order = rest
    return keep


def ap_from_pr(precision, recall):
    precision = np.array(precision, dtype=float)
    recall = np.array(recall, dtype=float)
    prev = np.r_[0.0, recall[:-1]]
    return float(np.sum(precision * (recall - prev)))


def match_mean_iou(pred_boxes, true_boxes):
    pred_boxes = [np.array(b, dtype=float) for b in pred_boxes]
    true_boxes = [np.array(b, dtype=float) for b in true_boxes]
    if not pred_boxes or not true_boxes:
        return 0.0
    scores = []
    used = set()
    for true_box in true_boxes:
        best = 0.0
        best_idx = -1
        for idx, pred_box in enumerate(pred_boxes):
            if idx in used:
                continue
            score = iou(pred_box, true_box)
            if score > best:
                best = score
                best_idx = idx
        if best_idx >= 0:
            used.add(best_idx)
        scores.append(best)
    return float(np.mean(scores))


def make_scene(size, true_boxes, pred_boxes, scores, seed):
    image = np.zeros((size, size), dtype=float)
    yy, xx = np.mgrid[0:size, 0:size]
    for idx, box in enumerate(true_boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        image[y1:y2, x1:x2] = 0.45 + 0.1 * idx
    noise = np.random.default_rng(seed).normal(0.0, 0.035, size=(size, size))
    image = np.clip(image + noise, 0.0, 1.0)
    return {
        "image": image,
        "true_boxes": np.array(true_boxes, dtype=float),
        "pred_boxes": np.array(pred_boxes, dtype=float),
        "scores": np.array(scores, dtype=float),
    }


def load_detection_ladder():
    rungs = []
    rungs.append(("D1 tiny hand scene", make_scene(8, [[0, 0, 3, 3]], [[1, 1, 4, 4]], [0.9], 1)))
    rungs.append(("D2 two clean boxes", make_scene(16, [[1, 2, 6, 8], [10, 9, 14, 14]], [[1, 2, 6, 8], [9, 8, 14, 14], [0, 1, 6, 7]], [0.95, 0.72, 0.45], 2)))
    rungs.append(("D3 crowded small boxes", make_scene(24, [[2, 2, 7, 8], [9, 3, 15, 9], [15, 14, 21, 21]], [[2, 2, 7, 8], [8, 3, 15, 10], [14, 13, 22, 21], [3, 3, 8, 9]], [0.93, 0.82, 0.74, 0.50], 3)))
    rungs.append(("D4 occlusion and scale", make_scene(32, [[2, 3, 8, 10], [9, 6, 18, 18], [19, 4, 29, 12], [21, 20, 30, 29]], [[1, 3, 9, 11], [8, 6, 18, 17], [18, 3, 30, 13], [20, 18, 31, 30], [10, 7, 18, 18]], [0.88, 0.84, 0.70, 0.66, 0.42], 4)))
    rungs.append(("D5 many noisy overlaps", make_scene(40, [[2, 2, 8, 9], [7, 6, 15, 17], [15, 4, 24, 13], [26, 5, 35, 16], [5, 24, 15, 35], [22, 23, 36, 36]], [[1, 1, 9, 10], [6, 5, 16, 18], [14, 5, 25, 14], [24, 4, 36, 17], [6, 23, 16, 35], [20, 21, 37, 37], [7, 7, 15, 17], [25, 5, 34, 16]], [0.86, 0.81, 0.75, 0.69, 0.58, 0.54, 0.62, 0.51], 5)))
    return rungs


def draw_boxes(ax, scene, pred_boxes, title):
    ax.imshow(scene["image"], cmap="gray", vmin=0.0, vmax=1.0)
    for box in scene["true_boxes"]:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
    for box in pred_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linestyle="--", linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def detect_and_score():
    a = np.array([0, 0, 3, 3], dtype=float)
    b = np.array([1, 1, 4, 4], dtype=float)
    overlap = iou(a, b)
    gt = np.array([1, 1, 3, 3], dtype=float)
    anchors = np.array([[0, 0, 2, 2], [0, 0, 3, 3], [1, 1, 4, 4]], dtype=float)
    anchor_ious = np.array([iou(anchor, gt) for anchor in anchors])
    boxes = np.array([[0, 0, 3, 3], [0.5, 0.5, 3.5, 3.5], [5, 5, 7, 7]], dtype=float)
    scores = np.array([0.9, 0.8, 0.7], dtype=float)
    kept = nms(boxes, scores, threshold=0.3)
    ap = ap_from_pr([1.0, 0.75, 0.6], [0.33, 0.67, 1.0])
    return overlap, anchor_ious, kept, ap

overlap, anchor_ious, kept, ap = detect_and_score()
print("IoU", round(overlap, 3))
print("anchor IoUs", np.round(anchor_ious, 3))
print("kept", kept)
print("AP", round(ap, 3))
assert round(overlap, 3) == 0.286
assert round(anchor_ious[1], 3) == 0.444
assert kept == [0, 2]
assert round(ap, 3) == 0.783

The reusable detector first sorts by confidence, suppresses boxes by union IoU, and then scores remaining boxes against ground truth.

In [ ]:
def run_scene(scene):
    keep = nms(scene["pred_boxes"], scene["scores"], threshold=0.3, sort=True)
    pred_boxes = scene["pred_boxes"][keep]
    metric = match_mean_iou(pred_boxes, scene["true_boxes"])
    return pred_boxes, metric

## Synthetic geometry ladder

These detection scenes replace the image-classification ladder because the topic is about boxes, overlap, and ranking. D1 is tiny and hand-checkable; D5 has many boxes, occlusion, and noisy duplicates.

In [ ]:
def box_area(box):
    w = max(0.0, float(box[2] - box[0]))
    h = max(0.0, float(box[3] - box[1]))
    return w * h


def iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = box_area([x1, y1, x2, y2])
    union = box_area(a) + box_area(b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def nms(boxes, scores, threshold=0.3, sort=True):
    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float)
    if sort:
        order = list(np.argsort(-scores))
    else:
        order = list(range(len(scores)))
    keep = []
    while order:
        current = order.pop(0)
        keep.append(int(current))
        rest = []
        for idx in order:
            overlap = iou(boxes[current], boxes[idx])
            if overlap <= threshold:
                rest.append(idx)
        order = rest
    return keep


def ap_from_pr(precision, recall):
    precision = np.array(precision, dtype=float)
    recall = np.array(recall, dtype=float)
    prev = np.r_[0.0, recall[:-1]]
    return float(np.sum(precision * (recall - prev)))


def match_mean_iou(pred_boxes, true_boxes):
    pred_boxes = [np.array(b, dtype=float) for b in pred_boxes]
    true_boxes = [np.array(b, dtype=float) for b in true_boxes]
    if not pred_boxes or not true_boxes:
        return 0.0
    scores = []
    used = set()
    for true_box in true_boxes:
        best = 0.0
        best_idx = -1
        for idx, pred_box in enumerate(pred_boxes):
            if idx in used:
                continue
            score = iou(pred_box, true_box)
            if score > best:
                best = score
                best_idx = idx
        if best_idx >= 0:
            used.add(best_idx)
        scores.append(best)
    return float(np.mean(scores))


def make_scene(size, true_boxes, pred_boxes, scores, seed):
    image = np.zeros((size, size), dtype=float)
    yy, xx = np.mgrid[0:size, 0:size]
    for idx, box in enumerate(true_boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        image[y1:y2, x1:x2] = 0.45 + 0.1 * idx
    noise = np.random.default_rng(seed).normal(0.0, 0.035, size=(size, size))
    image = np.clip(image + noise, 0.0, 1.0)
    return {
        "image": image,
        "true_boxes": np.array(true_boxes, dtype=float),
        "pred_boxes": np.array(pred_boxes, dtype=float),
        "scores": np.array(scores, dtype=float),
    }


def load_detection_ladder():
    rungs = []
    rungs.append(("D1 tiny hand scene", make_scene(8, [[0, 0, 3, 3]], [[1, 1, 4, 4]], [0.9], 1)))
    rungs.append(("D2 two clean boxes", make_scene(16, [[1, 2, 6, 8], [10, 9, 14, 14]], [[1, 2, 6, 8], [9, 8, 14, 14], [0, 1, 6, 7]], [0.95, 0.72, 0.45], 2)))
    rungs.append(("D3 crowded small boxes", make_scene(24, [[2, 2, 7, 8], [9, 3, 15, 9], [15, 14, 21, 21]], [[2, 2, 7, 8], [8, 3, 15, 10], [14, 13, 22, 21], [3, 3, 8, 9]], [0.93, 0.82, 0.74, 0.50], 3)))
    rungs.append(("D4 occlusion and scale", make_scene(32, [[2, 3, 8, 10], [9, 6, 18, 18], [19, 4, 29, 12], [21, 20, 30, 29]], [[1, 3, 9, 11], [8, 6, 18, 17], [18, 3, 30, 13], [20, 18, 31, 30], [10, 7, 18, 18]], [0.88, 0.84, 0.70, 0.66, 0.42], 4)))
    rungs.append(("D5 many noisy overlaps", make_scene(40, [[2, 2, 8, 9], [7, 6, 15, 17], [15, 4, 24, 13], [26, 5, 35, 16], [5, 24, 15, 35], [22, 23, 36, 36]], [[1, 1, 9, 10], [6, 5, 16, 18], [14, 5, 25, 14], [24, 4, 36, 17], [6, 23, 16, 35], [20, 21, 37, 37], [7, 7, 15, 17], [25, 5, 34, 16]], [0.86, 0.81, 0.75, 0.69, 0.58, 0.54, 0.62, 0.51], 5)))
    return rungs


def draw_boxes(ax, scene, pred_boxes, title):
    ax.imshow(scene["image"], cmap="gray", vmin=0.0, vmax=1.0)
    for box in scene["true_boxes"]:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
    for box in pred_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linestyle="--", linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def run_scene(scene):
    keep = nms(scene["pred_boxes"], scene["scores"], threshold=0.3, sort=True)
    pred_boxes = scene["pred_boxes"][keep]
    metric = match_mean_iou(pred_boxes, scene["true_boxes"])
    return pred_boxes, metric

rungs = load_detection_ladder()
for name, scene in rungs:
    print(name, "image", scene["image"].shape, "truth", len(scene["true_boxes"]), "pred", len(scene["pred_boxes"]))

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (name, scene) in zip(axes, rungs):
    draw_boxes(ax, scene, scene["pred_boxes"], name.split()[0])
plt.tight_layout()
plt.show()

## Run the same method across D1–D5

Each rung reports mean best-match IoU.

In [ ]:
results = []
outputs = []
for name, scene in rungs:
    pred_boxes, metric = run_scene(scene)
    results.append((name, metric))
    outputs.append(pred_boxes)

print("rung                         metric")
for name, metric in results:
    print(f"{name:28s} {metric:.3f}")

## Results visualization

Green boxes are truth, dashed red boxes are the method output. The curve shows localization quality as complexity rises.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (name, scene), pred_boxes in zip(axes[0], rungs, outputs):
    draw_boxes(ax, scene, pred_boxes, name.split()[0])

xs = np.arange(1, 6)
ys = [metric for name, metric in results]
axes[1, 0].plot(xs, ys, marker="o")
axes[1, 0].set_xticks(xs)
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].set_xlabel("complexity rung")
axes[1, 0].set_ylabel("IoU/AP metric")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5

Using intersection over predicted area and skipping score sorting can let weak duplicates survive while hiding oversized boxes. The fix is union IoU plus score-sorted NMS.

In [ ]:
name, scene = rungs[-1]
wrong_order = [6, 7, 2, 4, 5]
wrong_boxes = scene["pred_boxes"][wrong_order]
wrong_metric = match_mean_iou(wrong_boxes, scene["true_boxes"])
right_boxes, right_metric = run_scene(scene)
print("weak-first boxes", wrong_order)
print("wrong metric", round(wrong_metric, 3))
print("sorted union-IoU metric", round(right_metric, 3))

## Evaluate it + Practice

- Compare the reported IoU with a no-skill baseline that predicts one large center box or all background.
- Overfit D1: the hand scene should reproduce the exact lesson arithmetic before scaling up.
- Ablate the key idea, such as sorting before NMS, refinement, objectness, focal weighting, matching, or nearest-neighbor masks.
- Watch for failure signals: duplicate boxes, high pixel accuracy with poor IoU, and metrics that improve only because the scene got easier.

Practice:
1. Change the D3 object positions and predict how IoU changes.
2. Tighten the matching threshold and rerun the table.
3. Add one noisy false positive to D5 and explain the curve.